# Install Library

In [11]:
pip install enoppy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.8/168.8 kB 9.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.5/58.5 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.3/423.3 kB 22.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.9/17.9 MB 70.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.0/13.0 MB 67.7 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.0 which is incompatible.
shap 0.51.0 requires numpy>=2, but you have numpy 1.26.0 which is incompatible.
mu

# Import Library

In [4]:
import os
import json
import numpy as np
import pandas as pd

# Connected to Drive

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Helper Functions

In [6]:
def load_algorithm_results(algo):
    fit_path = os.path.join(ROOT, algo, "best_fit", f"{algo}_best_fit.csv")
    pos_path = os.path.join(ROOT, algo, "best_position", f"{algo}_best_position.csv")

    fit_df = pd.read_csv(fit_path)
    pos_df = pd.read_csv(pos_path)

    return fit_df, pos_df


def parse_position(pos_str):
    if isinstance(pos_str, str):
        return np.array(json.loads(pos_str), dtype=float)
    return np.array(pos_str, dtype=float)

In [7]:
def build_optimization_table(problem_name):
    rows = []

    for algo in ALGORITHMS:
        fit_df, pos_df = load_algorithm_results(algo)

        fitness_values = fit_df[problem_name].astype(float).values
        best_idx = np.argmin(fitness_values)

        best_fitness = fitness_values[best_idx]
        mean_fitness = np.mean(fitness_values)
        std_fitness = np.std(fitness_values, ddof=1)

        best_position = parse_position(pos_df.loc[best_idx, problem_name])

        row = {
            "Algorithm": algo,
            "Best Fitness": best_fitness,
            "Mean": mean_fitness,
            "Std": std_fitness,
        }

        for i, val in enumerate(best_position, start=1):
            row[f"x{i}"] = val

        rows.append(row)

    table = pd.DataFrame(rows)

    table["Rank"] = table["Best Fitness"].rank(method="min", ascending=True).astype(int)
    table = table.sort_values("Rank").reset_index(drop=True)

    return table

In [16]:
def calculate_max_constraint_violation(problem, solution):
    """
    Feasibility metric:
    max(0, max(g(x)))
    """

    solution = np.array(solution, dtype=float)

    if hasattr(problem, "get_cons"):
        g = problem.get_cons(solution)

    else:
        raise AttributeError(
            f"{type(problem).__name__} için get_cons metodu bulunamadı."
        )

    g = np.array(g, dtype=float).reshape(-1)

    return float(max(0.0, np.max(g)))

In [17]:
def build_feasibility_table(problem_name, problem_object):
    rows = []

    for algo in ALGORITHMS:
        fit_df, pos_df = load_algorithm_results(algo)

        fitness_values = fit_df[problem_name].astype(float).values
        best_idx = np.argmin(fitness_values)

        best_fitness = fitness_values[best_idx]
        best_position = parse_position(pos_df.loc[best_idx, problem_name])

        max_violation = calculate_max_constraint_violation(
            problem_object,
            best_position
        )

        rows.append({
            "Algorithm": algo,
            "Problem": problem_name,
            "Best Fitness": best_fitness,
            "Max Constraint Violation": max_violation,
            "Feasible": max_violation <= 1e-8
        })

    table = pd.DataFrame(rows)
    table["Rank"] = table["Best Fitness"].rank(method="min", ascending=True).astype(int)
    table = table.sort_values("Rank").reset_index(drop=True)

    return table

# Get Results

In [18]:
from enoppy.paper_based.rwco_2020 import (
    PressureVesselDesignProblem,
    WeldedBeamDesignProblem,
    ThreeBarTrussDesignProblem,
    MultipleDiskClutchBrakeDesignProblem,
)

from enoppy.paper_based.pdo_2022 import (
    TubularColumnProblem,
    CorrugatedBulkheadProblem,
)

In [22]:
problems = [
    PressureVesselDesignProblem(),
    WeldedBeamDesignProblem(),
    ThreeBarTrussDesignProblem(),
    MultipleDiskClutchBrakeDesignProblem(),
    TubularColumnProblem(),
    CorrugatedBulkheadProblem(),
]

names = [
    "Pressure Vessel",
    "Welded Beam",
    "Three Bar Truss",
    "Multiple Disk",
    "Tubular-Column",
    "Corrugated-Bulkhead",
]

In [23]:
ROOT = "/content/drive/My Drive/Revision/history/engineering"

ALGORITHMS = [
    "NHO", "ACO", "DE", "GA", "GWO", "HGSO", "HHO", "SSO", "ACSA",
    "BPBO", "CHO", "SRA", "L_SHADE", "IMODE", "LSHADEcnEpSin"
]

OUTPUT_DIR = os.path.join(ROOT, "_summary_tables")
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [24]:
# Notebook içinde zaten tanımlı olmalı:
# names, functions, problems = engineering_problems()

for problem_name, problem_object in zip(names, problems):

    opt_table = build_optimization_table(problem_name)
    feas_table = build_feasibility_table(problem_name, problem_object)

    safe_name = problem_name.replace(" ", "_").replace("-", "_")

    opt_table.to_csv(
        os.path.join(OUTPUT_DIR, f"{safe_name}_optimization_results.csv"),
        index=False
    )

    feas_table.to_csv(
        os.path.join(OUTPUT_DIR, f"{safe_name}_feasibility_audit.csv"),
        index=False
    )

    opt_table.to_latex(
        os.path.join(OUTPUT_DIR, f"{safe_name}_optimization_results.tex"),
        index=False,
        float_format="%.4e"
    )

    feas_table.to_latex(
        os.path.join(OUTPUT_DIR, f"{safe_name}_feasibility_audit.tex"),
        index=False,
        float_format="%.4e"
    )

print("All optimization and feasibility tables were saved to:")
print(OUTPUT_DIR)

All optimization and feasibility tables were saved to:
/content/drive/My Drive/Revision/history/engineering/_summary_tables


# Version 2


In [42]:
import os
import json
import numpy as np
import pandas as pd

# ============================================================
# 1. PATH SETTINGS
# ============================================================

ROOT = "/content/drive/My Drive/Revision/history/engineering"

OUTPUT_DIR = os.path.join(ROOT, "_summary_tables_v2_raw_feasibility")
os.makedirs(OUTPUT_DIR, exist_ok=True)

ALGORITHMS = [
    "NHO", "ACO", "DE", "GA", "GWO", "HGSO", "HHO", "SSO",
    "ACSA", "BPBO", "CHO", "SRA",
    "L_SHADE", "IMODE", "LSHADEcnEpSin"
]

TOL = 1e-8

In [43]:
# ============================================================
# 2. LOAD SAVED RESULTS
# ============================================================

def load_algorithm_results(algo):
    fit_path = os.path.join(ROOT, algo, "best_fit", f"{algo}_best_fit.csv")
    pos_path = os.path.join(ROOT, algo, "best_position", f"{algo}_best_position.csv")

    if not os.path.exists(fit_path):
        raise FileNotFoundError(f"Best fitness file not found: {fit_path}")

    if not os.path.exists(pos_path):
        raise FileNotFoundError(f"Best position file not found: {pos_path}")

    fit_df = pd.read_csv(fit_path)
    pos_df = pd.read_csv(pos_path)

    return fit_df, pos_df


def parse_position(pos):
    if isinstance(pos, str):
        return np.array(json.loads(pos), dtype=float)
    return np.array(pos, dtype=float)

In [44]:
# ============================================================
# 3. DEFINE ENGINEERING PROBLEMS
# ============================================================

from enoppy.paper_based.rwco_2020 import (
    PressureVesselDesignProblem,
    WeldedBeamDesignProblem,
    ThreeBarTrussDesignProblem,
    MultipleDiskClutchBrakeDesignProblem,
)

from enoppy.paper_based.pdo_2022 import (
    TubularColumnProblem,
    CorrugatedBulkheadProblem,
)

problem_names = [
    "Pressure Vessel",
    "Welded Beam",
    "Three Bar Truss",
    "Multiple Disk",
    "Tubular-Column",
    "Corrugated-Bulkhead",
]

problem_objects = [
    PressureVesselDesignProblem(),
    WeldedBeamDesignProblem(),
    ThreeBarTrussDesignProblem(),
    MultipleDiskClutchBrakeDesignProblem(),
    TubularColumnProblem(),
    CorrugatedBulkheadProblem(),
]

In [45]:
# ============================================================
# 4. CONSTRAINT VIOLATION FUNCTIONS
# ============================================================

def amend_if_needed(problem, solution):
    """
    Applies problem-specific amend_position if available.
    This is important for problems with integer/discrete variables,
    such as Pressure Vessel and Multiple Disk Clutch Brake.
    """

    x = parse_position(solution).copy()

    if hasattr(problem, "amend_position"):
        try:
            x = problem.amend_position(x, problem.lb, problem.ub)
        except Exception:
            try:
                x = problem.amend_position(x)
            except Exception:
                pass

    return np.array(x, dtype=float)


def get_raw_constraints(problem, solution, apply_amend=True):
    """
    Returns raw constraint values g(x).

    Feasibility convention:
        g(x) <= 0
    """

    if apply_amend:
        x = amend_if_needed(problem, solution)
    else:
        x = parse_position(solution).copy()

    if not hasattr(problem, "get_cons"):
        raise AttributeError(f"{type(problem).__name__} has no get_cons() method.")

    g = np.array(problem.get_cons(x), dtype=float).reshape(-1)

    return g


def calculate_constraint_violation(problem, solution, tol=1e-8, apply_amend=True):
    """
    Calculates raw and thresholded constraint violation.

    Raw Max g(x):
        max(g(x))

    Max Constraint Violation:
        max(0, max(g(x)))

    If violation <= tol, it is treated as feasible and reported as 0.0.
    """

    g = get_raw_constraints(
        problem=problem,
        solution=solution,
        apply_amend=apply_amend
    )

    raw_max_g = float(np.max(g))
    raw_violation = max(0.0, raw_max_g)

    feasible = raw_violation <= tol

    reported_violation = 0.0 if feasible else raw_violation

    return {
        "Raw Max g(x)": raw_max_g,
        "Raw Violation": raw_violation,
        "Max Constraint Violation": reported_violation,
        "Feasible": feasible,
        "All g(x)": g
    }


def feasibility_statistics(problem, positions, tol=1e-8, apply_amend=True):
    """
    Calculates feasibility statistics across all independent runs.
    """

    raw_max_values = []
    raw_violations = []
    reported_violations = []
    feasible_flags = []

    for pos in positions:
        result = calculate_constraint_violation(
            problem=problem,
            solution=pos,
            tol=tol,
            apply_amend=apply_amend
        )

        raw_max_values.append(result["Raw Max g(x)"])
        raw_violations.append(result["Raw Violation"])
        reported_violations.append(result["Max Constraint Violation"])
        feasible_flags.append(result["Feasible"])

    raw_max_values = np.array(raw_max_values, dtype=float)
    raw_violations = np.array(raw_violations, dtype=float)
    reported_violations = np.array(reported_violations, dtype=float)
    feasible_flags = np.array(feasible_flags, dtype=bool)

    return {
        "Raw Max g(x) Mean": float(np.mean(raw_max_values)),
        "Raw Max g(x) Max": float(np.max(raw_max_values)),
        "Raw Violation Mean": float(np.mean(raw_violations)),
        "Raw Violation Max": float(np.max(raw_violations)),
        "Mean Constraint Violation": float(np.mean(reported_violations)),
        "Max Constraint Violation Across Runs": float(np.max(reported_violations)),
        "Feasible Run Ratio (%)": float(100.0 * np.mean(feasible_flags)),
    }

In [46]:
# ============================================================
# 5. BUILD SUMMARY AND SOLUTION TABLES
# ============================================================

def build_engineering_summary_table(problem_name, problem_object, tol=1e-8):
    """
    Main summary table.

    Columns:
    Algorithm
    Best Fitness
    Mean
    Std
    Rank
    Best Raw Max g(x)
    Best Max Constraint Violation
    Mean Constraint Violation
    Feasible Run Ratio (%)
    Best Solution Feasible
    """

    rows = []

    for algo in ALGORITHMS:

        fit_df, pos_df = load_algorithm_results(algo)

        if problem_name not in fit_df.columns:
            raise KeyError(f"{problem_name} not found in fitness file for {algo}")

        if problem_name not in pos_df.columns:
            raise KeyError(f"{problem_name} not found in position file for {algo}")

        fitness_values = fit_df[problem_name].astype(float).values
        positions = pos_df[problem_name].values

        best_idx = int(np.argmin(fitness_values))

        best_fitness = float(fitness_values[best_idx])
        mean_fitness = float(np.mean(fitness_values))
        std_fitness = float(np.std(fitness_values, ddof=1))

        best_position = positions[best_idx]

        best_result = calculate_constraint_violation(
            problem=problem_object,
            solution=best_position,
            tol=tol,
            apply_amend=True
        )

        stats = feasibility_statistics(
            problem=problem_object,
            positions=positions,
            tol=tol,
            apply_amend=True
        )

        rows.append({
            "Algorithm": algo,
            "Best Fitness": best_fitness,
            "Mean": mean_fitness,
            "Std": std_fitness,
            "Best Raw Max g(x)": best_result["Raw Max g(x)"],
            "Best Raw Violation": best_result["Raw Violation"],
            "Best Max Constraint Violation": best_result["Max Constraint Violation"],
            "Mean Constraint Violation": stats["Mean Constraint Violation"],
            "Max Constraint Violation Across Runs": stats["Max Constraint Violation Across Runs"],
            "Raw Max g(x) Mean": stats["Raw Max g(x) Mean"],
            "Raw Max g(x) Max": stats["Raw Max g(x) Max"],
            "Feasible Run Ratio (%)": stats["Feasible Run Ratio (%)"],
            "Best Solution Feasible": "Yes" if best_result["Feasible"] else "No"
        })

    table = pd.DataFrame(rows)

    table["Rank"] = table["Best Fitness"].rank(
        method="min",
        ascending=True
    ).astype(int)

    table = table[
        [
            "Algorithm",
            "Best Fitness",
            "Mean",
            "Std",
            "Rank",
            "Best Raw Max g(x)",
            "Best Raw Violation",
            "Best Max Constraint Violation",
            "Mean Constraint Violation",
            "Max Constraint Violation Across Runs",
            "Raw Max g(x) Mean",
            "Raw Max g(x) Max",
            "Feasible Run Ratio (%)",
            "Best Solution Feasible"
        ]
    ]

    table = table.sort_values("Rank").reset_index(drop=True)

    return table


def build_solution_table(problem_name):
    """
    Solution table:
    Algorithm | x1 | x2 | ... | xn
    """

    rows = []

    for algo in ALGORITHMS:

        fit_df, pos_df = load_algorithm_results(algo)

        fitness_values = fit_df[problem_name].astype(float).values
        best_idx = int(np.argmin(fitness_values))

        best_position = parse_position(pos_df.loc[best_idx, problem_name])

        row = {"Algorithm": algo}

        for i, val in enumerate(best_position, start=1):
            row[f"x{i}"] = float(val)

        rows.append(row)

    table = pd.DataFrame(rows)

    return table

In [47]:
# ============================================================
# 6. BUILD COMPACT TABLE FOR MANUSCRIPT
# ============================================================

def build_manuscript_summary_table(full_table):
    """
    Compact table for manuscript use.
    Keeps only the columns that are useful in the paper.
    """

    compact = full_table[
        [
            "Algorithm",
            "Best Fitness",
            "Mean",
            "Std",
            "Rank",
            "Best Max Constraint Violation",
            "Mean Constraint Violation",
            "Feasible Run Ratio (%)",
            "Best Solution Feasible"
        ]
    ].copy()

    return compact

In [49]:
# ============================================================
# 7. SAVE ALL TABLES
# ============================================================

all_problem_summary = []

for problem_name, problem_object in zip(problem_names, problem_objects):

    print(f"Processing: {problem_name}")

    full_summary_table = build_engineering_summary_table(
        problem_name=problem_name,
        problem_object=problem_object,
        tol=TOL
    )

    manuscript_summary_table = build_manuscript_summary_table(
        full_summary_table
    )

    solution_table = build_solution_table(problem_name)

    safe_name = problem_name.replace(" ", "_").replace("-", "_")

    full_summary_table.to_csv(
        os.path.join(OUTPUT_DIR, f"{safe_name}_full_feasibility_diagnostics.csv"),
        index=False
    )

    manuscript_summary_table.to_csv(
        os.path.join(OUTPUT_DIR, f"{safe_name}_manuscript_summary.csv"),
        index=False
    )

    solution_table.to_csv(
        os.path.join(OUTPUT_DIR, f"{safe_name}_solutions.csv"),
        index=False
    )

    manuscript_summary_table.to_latex(
        os.path.join(OUTPUT_DIR, f"{safe_name}_manuscript_summary.tex"),
        index=False,
        float_format="%.4f"
    )

    solution_table.to_latex(
        os.path.join(OUTPUT_DIR, f"{safe_name}_solutions.tex"),
        index=False,
        float_format="%.4f"
    )

    temp = manuscript_summary_table.copy()
    temp.insert(0, "Problem", problem_name)
    all_problem_summary.append(temp)

all_problem_summary_df = pd.concat(all_problem_summary, ignore_index=True)

all_problem_summary_df.to_csv(
    os.path.join(OUTPUT_DIR, "ALL_PROBLEMS_manuscript_summary.csv"),
    index=False
)

print("\nAll updated feasibility and solution tables were saved to:")
print(OUTPUT_DIR)

Processing: Pressure Vessel
Processing: Welded Beam
Processing: Three Bar Truss
Processing: Multiple Disk
Processing: Tubular-Column
Processing: Corrugated-Bulkhead

All updated feasibility and solution tables were saved to:
/content/drive/My Drive/Revision/history/engineering/_summary_tables_v2_raw_feasibility
